# Laboratorium terbuka: model satu spesies

Notebook ini merupakan pendamping komputasi mandiri untuk Bab 5. Seluruh contoh memakai Python terbuka dan dapat dijalankan secara luring setelah paket pada requirements.lock dipasang. Notebook tidak memerlukan layanan web saat dijalankan.

Tujuh eksperimen berikut membandingkan waktu diskret dan kontinu, mengamati penggandaan periode dan chaos, memeriksa solusi logistik, serta mensimulasikan model berstruktur umur dan model LPA. Eksperimen terakhir memakai salinan lokal berhash dari data resmi Biro Sensus Amerika Serikat yang dirujuk dalam Soal 16.

**ID unit:** O005-LEGA-V101-CH05  
**ID notebook:** O005-LEGA-V101-CH05-NB01  
**Lisensi notebook:** CC BY-NC-SA 4.0  
**Asal komponen:** pendamping komputasi baru berdasarkan persamaan dan soal Bab 5; bukan salinan kode MATLAB atau Excel.

## 1. Pertumbuhan eksponensial diskret dan kontinu

Model diskret $N_{k+1}=\kappa N_k$ memiliki solusi $N_k=\kappa^kN_0$. Jika $\kappa=1+r\Delta t$, model kontinu terkait adalah $dN/dt=rN$ dengan solusi $N(t)=N_0e^{rt}$. Keduanya berdekatan untuk $r\Delta t$ kecil, tetapi tidak identik.

In [ ]:
import os
import re
import hashlib
from pathlib import Path

import numpy as np
import scipy
from scipy.integrate import solve_ivp
import matplotlib

if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

N0 = 100.0
r = 0.025
delta_t = 1.0
kappa = 1.0 + r * delta_t
tahun = np.arange(0, 41, dtype=float)
N_diskret = N0 * kappa ** tahun
N_kontinu = N0 * np.exp(r * tahun)
selisih_relatif_akhir = (N_kontinu[-1] - N_diskret[-1]) / N_kontinu[-1]

assert np.isclose(N_diskret[0], N0)
assert np.allclose(N_diskret[1:], kappa * N_diskret[:-1])
assert np.all(N_diskret > 0.0) and np.all(N_kontinu > 0.0)
assert np.all(N_kontinu >= N_diskret)
assert 0.0 < selisih_relatif_akhir < 0.02

fig, ax = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)
ax.plot(tahun, N_diskret, "o-", markersize=3.5, label="diskret")
ax.plot(tahun, N_kontinu, "--", linewidth=2.2, label="kontinu")
ax.set(xlabel="waktu (tahun)", ylabel="N", title="Pertumbuhan eksponensial: diskret dan kontinu")
ax.grid(alpha=0.25)
ax.legend()
plt.show()
print(f"Python/NumPy/SciPy/Matplotlib: {np.__version__} / {scipy.__version__} / {matplotlib.__version__}")
print(f"N_diskret(40)={N_diskret[-1]:.6f}; N_kontinu(40)={N_kontinu[-1]:.6f}; selisih relatif={selisih_relatif_akhir:.6%}")

## 2. Titik tetap, diagram sarang laba-laba, dan penggandaan periode

Untuk peta logistik $x_{n+1}=a x_n(1-x_n)$, titik tetapnya adalah $0$ dan, jika $a>1$, $1-1/a$. Suatu titik tetap stabil secara linear jika nilai mutlak turunan peta di titik tersebut kurang dari satu. Iterasi jangka panjang memperlihatkan siklus periode 1, 2, 4, lalu 8 ketika $a$ dinaikkan melalui nilai contoh di bawah.

In [ ]:
def orbit_logistik(a, x0, langkah):
    x = np.empty(langkah + 1, dtype=float)
    x[0] = x0
    for n in range(langkah):
        x[n + 1] = a * x[n] * (1.0 - x[n])
    return x

a_uji = 2.8
titik_tetap = np.array([0.0, 1.0 - 1.0 / a_uji])
turunan = a_uji * (1.0 - 2.0 * titik_tetap)
assert np.allclose(a_uji * titik_tetap * (1.0 - titik_tetap), titik_tetap)
assert abs(turunan[0]) > 1.0 and abs(turunan[1]) < 1.0

nilai_a = (2.8, 3.2, 3.5, 3.55)
periode = {}
for a in nilai_a:
    ekor = orbit_logistik(a, 0.173, 3000)[-512:]
    periode[a] = len(np.unique(np.round(ekor, 8)))
assert tuple(periode.values()) == (1, 2, 4, 8)

a_sarang = 3.2
lintasan = orbit_logistik(a_sarang, 0.173, 18)
x_grid = np.linspace(0.0, 1.0, 600)
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11.2, 4.8), constrained_layout=True)
ax0.plot(x_grid, a_sarang * x_grid * (1.0 - x_grid), label="$f(x)$")
ax0.plot(x_grid, x_grid, color="#555555", linewidth=1.2, label="$y=x$")
for x_n, x_baru in zip(lintasan[:-1], lintasan[1:]):
    ax0.plot([x_n, x_n], [x_n, x_baru], color="#D55E00", linewidth=0.9)
    ax0.plot([x_n, x_baru], [x_baru, x_baru], color="#D55E00", linewidth=0.9)
ax0.set(xlim=(0, 1), ylim=(0, 1), xlabel="$x_n$", ylabel="$x_{n+1}$", title="Sarang laba-laba, a=3,2")
ax0.legend()

for a in np.linspace(2.8, 4.0, 360):
    ekor = orbit_logistik(float(a), 0.173, 1800)[-90:]
    ax1.plot(np.full_like(ekor, a), ekor, ",", color="#0072B2", alpha=0.65)
ax1.set(xlabel="$a$", ylabel="atraktor $x$", title="Diagram bifurkasi numerik")
plt.show()
print("Perkiraan periode atraktor:", periode)

## 3. Eksponen Lyapunov

Eksponen Lyapunov numerik $\lambda_L=\lim n^{-1}\sum\log|f'(x_k)|$ mengukur pemisahan gangguan kecil. Nilai negatif menunjukkan kontraksi menuju atraktor periodik, sedangkan nilai positif menunjukkan sensitivitas eksponensial terhadap kondisi awal. Untuk $a=4$, nilai teoritisnya adalah $\log 2$ untuk hampir semua kondisi awal.

In [ ]:
def eksponen_lyapunov(a, x0=0.173, buang=2000, ambil=12000):
    x = float(x0)
    for _ in range(buang):
        x = a * x * (1.0 - x)
    jumlah = 0.0
    for _ in range(ambil):
        turunan_abs = max(abs(a * (1.0 - 2.0 * x)), 1e-300)
        jumlah += np.log(turunan_abs)
        x = a * x * (1.0 - x)
    return jumlah / ambil

lambda_32 = eksponen_lyapunov(3.2)
lambda_39 = eksponen_lyapunov(3.9)
lambda_40 = eksponen_lyapunov(4.0)
assert lambda_32 < 0.0
assert lambda_39 > 0.3
assert abs(lambda_40 - np.log(2.0)) < 0.02

a_grid = np.linspace(2.5, 4.0, 260)
lambda_grid = np.array([eksponen_lyapunov(float(a), buang=1000, ambil=3500) for a in a_grid])
assert np.all(np.isfinite(lambda_grid))
assert np.any(lambda_grid > 0.0) and np.any(lambda_grid < 0.0)

fig, ax = plt.subplots(figsize=(8.4, 4.8), constrained_layout=True)
ax.plot(a_grid, lambda_grid, color="#009E73", linewidth=1.2)
ax.axhline(0.0, color="#555555", linewidth=1.0)
ax.set(xlabel="$a$", ylabel="eksponen Lyapunov", title="Jendela periodik dan wilayah chaos")
ax.grid(alpha=0.2)
plt.show()
print(f"lambda(3,2)={lambda_32:.6f}; lambda(3,9)={lambda_39:.6f}; lambda(4)={lambda_40:.6f}; log(2)={np.log(2):.6f}")

## 4. Solusi eksak dan numerik model logistik kontinu

Untuk $dM/dt=\lambda M(1-M)$ dan $M_0>0$, solusi eksaknya adalah $M(t)=[1+(1-M_0)M_0^{-1}e^{-\lambda t}]^{-1}$. Kondisi awal $M_0=0$ harus ditangani tersendiri: solusinya identik nol. Kita membandingkan kedua kasus dengan solve_ivp.

In [ ]:
def solusi_logistik_eksak(t, laju, M0):
    t = np.asarray(t, dtype=float)
    if M0 == 0.0:
        return np.zeros_like(t)
    return 1.0 / (1.0 + ((1.0 - M0) / M0) * np.exp(-laju * t))

def solusi_logistik_numerik(t, laju, M0):
    hasil = solve_ivp(
        lambda waktu, y: laju * y * (1.0 - y),
        (float(t[0]), float(t[-1])),
        [float(M0)],
        t_eval=t,
        rtol=1e-11,
        atol=1e-13,
    )
    assert hasil.success
    return hasil.y[0]

t_eval = np.linspace(0.0, 12.0, 401)
laju = 0.7
galat = {}
fig, ax = plt.subplots(figsize=(8.3, 4.8), constrained_layout=True)
for M0 in (0.0, 0.1, 1.5):
    eksak = solusi_logistik_eksak(t_eval, laju, M0)
    numerik = solusi_logistik_numerik(t_eval, laju, M0)
    galat[M0] = float(np.max(np.abs(eksak - numerik)))
    ax.plot(t_eval, numerik, label=f"M0={M0:g}")
assert galat[0.0] == 0.0
assert galat[0.1] < 1e-9 and galat[1.5] < 1e-9
assert np.all(solusi_logistik_eksak(t_eval, laju, 0.0) == 0.0)
assert np.isclose(solusi_logistik_eksak(t_eval, laju, 0.1)[-1], 1.0, atol=0.003)
ax.axhline(1.0, color="#555555", linewidth=1.0, linestyle="--")
ax.set(xlabel="waktu", ylabel="$M(t)$", title="Model logistik kontinu")
ax.legend()
ax.grid(alpha=0.2)
plt.show()
print("Galat maksimum solve_ivp terhadap solusi eksak:", galat)

## 5. Proyeksi tahap umur dan nilai eigen

Vektor $C=(N_1,N_2,N_3)^T$ diproyeksikan dengan matriks tak-negatif $A$. Radius spektral menentukan laju asimtotik dominan ketika vektor awal mempunyai komponen pada arah eigen utama. Parameter demonstrasi dipilih agar semua peluang per langkah berada pada rentang yang masuk akal.

In [ ]:
delta_t = 1.0
d1, g1 = 0.05, 0.25
d2, g2 = 0.04, 0.15
d3, b2 = 0.08, 0.35
A_tahap = np.array([
    [1.0 - (d1 + g1) * delta_t, b2 * delta_t, 0.0],
    [g1 * delta_t, 1.0 - (d2 + g2) * delta_t, 0.0],
    [0.0, g2 * delta_t, 1.0 - d3 * delta_t],
])
assert np.all(A_tahap >= 0.0)

keadaan = np.empty((41, 3), dtype=float)
keadaan[0] = [1000.0, 800.0, 500.0]
for n in range(40):
    keadaan[n + 1] = A_tahap @ keadaan[n]
assert np.all(keadaan >= 0.0)
assert np.allclose(keadaan[1:], keadaan[:-1] @ A_tahap.T)

nilai_eigen, vektor_eigen = np.linalg.eig(A_tahap)
indeks_utama = int(np.argmax(np.abs(nilai_eigen)))
r_utama = nilai_eigen[indeks_utama]
v_utama = vektor_eigen[:, indeks_utama]
residu_eigen = np.linalg.norm(A_tahap @ v_utama - r_utama * v_utama)
radius_spektral = float(abs(r_utama))
assert abs(r_utama.imag) < 1e-12
assert residu_eigen < 1e-12
assert radius_spektral > 1.0
assert keadaan[-1].sum() > keadaan[0].sum()

fig, ax = plt.subplots(figsize=(8.3, 4.8), constrained_layout=True)
for i, nama in enumerate(("kelompok 1", "kelompok 2", "kelompok 3")):
    ax.plot(np.arange(41), keadaan[:, i], label=nama)
ax.set(xlabel="langkah", ylabel="jumlah individu", title="Dinamika proyeksi tiga tahap")
ax.legend()
ax.grid(alpha=0.2)
plt.show()
print(f"Nilai eigen={nilai_eigen}; radius spektral={radius_spektral:.9f}; residu={residu_eigen:.3e}")

## 6. Simulasi deterministik model LPA

Model larva–pupa–dewasa (LPA) memasukkan kanibalisme telur oleh larva dan dewasa serta kanibalisme pupa oleh dewasa melalui faktor eksponensial. Semua parameter di sini merupakan nilai demonstrasi, bukan hasil kalibrasi terhadap eksperimen. Peta mempertahankan oktan tak-negatif.

In [ ]:
PARAMETER_LPA = {
    "b": 7.5,
    "d_l": 0.20,
    "d_a": 0.10,
    "c_ea": 0.010,
    "c_el": 0.005,
    "c_pa": 0.004,
}

def simulasi_lpa(awal, langkah, p=PARAMETER_LPA):
    hasil = np.empty((langkah + 1, 3), dtype=float)
    hasil[0] = np.asarray(awal, dtype=float)
    for n in range(langkah):
        L, P, A = hasil[n]
        hasil[n + 1, 0] = p["b"] * A * np.exp(-p["c_ea"] * A - p["c_el"] * L)
        hasil[n + 1, 1] = (1.0 - p["d_l"]) * L
        hasil[n + 1, 2] = (1.0 - p["d_a"]) * A + P * np.exp(-p["c_pa"] * A)
    return hasil

lpa = simulasi_lpa([20.0, 10.0, 15.0], 160)
lpa_ulangan = simulasi_lpa([20.0, 10.0, 15.0], 160)
assert np.array_equal(lpa, lpa_ulangan)
assert np.all(np.isfinite(lpa))
assert np.all(lpa >= 0.0)
assert np.max(lpa) < 1e6
assert np.all(lpa[-1] > 0.0) and np.linalg.norm(lpa[-1] - lpa[-2]) < 1e-10

fig, ax = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
for i, nama in enumerate(("larva", "pupa", "dewasa")):
    ax.plot(np.arange(lpa.shape[0]), lpa[:, i], label=nama)
ax.set(xlabel="langkah dua-mingguan", ylabel="jumlah individu", title="Model LPA deterministik")
ax.legend(ncol=3)
ax.grid(alpha=0.2)
plt.show()
print(f"Rentang 40 langkah terakhir (L,P,A): {np.ptp(lpa[-40:], axis=0)}")

## 7. Soal 16: data resmi populasi Amerika Serikat

Berkas lokal popclockest.txt adalah salinan byte-for-byte dari URL resmi Biro Sensus yang dicantumkan dalam bab. Hash SHA-256 yang diharapkan adalah f59dbd91b2bf975df7b7fb4af6de52dc3c68a705632e83d60410d98781206f09. Kode tidak mengunduh data saat dijalankan.

Catatan sumber membatasi perbandingan langsung: data 1900–1949 tidak mencakup Alaska dan Hawaii; cakupan Angkatan Bersenjata di luar negeri berubah menurut periode; dan nilai 1900–1929 dibulatkan ke ribuan terdekat. Karena itu, pencocokan eksponensial di bawah merupakan diagnosis model sederhana, bukan klaim bahwa definisi populasi seragam sepanjang abad.

In [ ]:
HASH_CENSUS = "f59dbd91b2bf975df7b7fb4af6de52dc3c68a705632e83d60410d98781206f09"
kandidat_data = [
    Path("../data/popclockest.txt"),
    Path("data/popclockest.txt"),
    Path("source/id-ID/O005-LEGA-V101-CH05/data/popclockest.txt"),
]
jalur_data = next((p.resolve() for p in kandidat_data if p.is_file()), None)
assert jalur_data is not None, "Salinan lokal popclockest.txt tidak ditemukan."
byte_data = jalur_data.read_bytes()
hash_aktual = hashlib.sha256(byte_data).hexdigest()
assert hash_aktual == HASH_CENSUS
teks_data = byte_data.decode("ascii")

baris = re.findall(r"^\s*July 1,\s+(\d{4})\s+([\d,]+)", teks_data, flags=re.MULTILINE)
tahun_census = np.array([int(tahun) for tahun, _ in baris], dtype=int)
populasi = np.array([int(nilai.replace(",", "")) for _, nilai in baris], dtype=float)
urutan = np.argsort(tahun_census)
tahun_census, populasi = tahun_census[urutan], populasi[urutan]
assert len(tahun_census) == 100
assert (tahun_census[0], tahun_census[-1]) == (1900, 1999)
assert np.all(np.diff(tahun_census) == 1)
assert populasi[0] == 76094000 and populasi[-1] == 272690813
assert "exclude the" in teks_data and "Alaska and Hawaii" in teks_data

kappa_ls = float(np.dot(populasi[:-1], populasi[1:]) / np.dot(populasi[:-1], populasi[:-1]))
R_ls = kappa_ls - 1.0
waktu = tahun_census - tahun_census[0]
laju_log, log_N0 = np.polyfit(waktu, np.log(populasi), 1)
prediksi = np.exp(log_N0 + laju_log * waktu)
rmse_relatif = float(np.sqrt(np.mean(((prediksi - populasi) / populasi) ** 2)))
assert 0.0 < R_ls < 0.03
assert 0.0 < laju_log < 0.03
assert rmse_relatif < 0.08

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11.4, 4.8), constrained_layout=True)
ax0.plot(tahun_census, populasi / 1e6, label="data resmi")
ax0.plot(tahun_census, prediksi / 1e6, "--", label="pencocokan eksponensial")
ax0.set(xlabel="tahun", ylabel="populasi (juta)", title="Populasi nasional, 1900–1999")
ax0.legend()
R_tahunan = populasi[1:] / populasi[:-1] - 1.0
ax1.plot(tahun_census[1:], 100.0 * R_tahunan, color="#D55E00")
ax1.axhline(100.0 * R_ls, color="#555555", linestyle="--", label="R kuadrat terkecil")
ax1.set(xlabel="tahun", ylabel="pertumbuhan satu tahun (%)", title="Laju pertumbuhan tidak konstan")
ax1.legend()
for ax in (ax0, ax1):
    ax.grid(alpha=0.2)
plt.show()
print(f"data={jalur_data.name}; bytes={len(byte_data)}; SHA-256={hash_aktual}")
print(f"R_ls={R_ls:.6%}; laju log={laju_log:.6%}; RMSE relatif={rmse_relatif:.6%}")

## Kesimpulan

Ketujuh sel komputasi memberi pemeriksaan yang dapat diulang untuk pertumbuhan linear, peta nonlinear, solusi ODE, proyeksi tahap, model LPA, dan data Soal 16. Grafik adalah alat diagnosis; kesesuaian visual tidak menggantikan pemeriksaan asumsi, definisi data, kestabilan, atau residu. Semua parameter yang tidak berasal dari data resmi diberi label sebagai nilai demonstrasi.